# 03-3. 동일 조건 LGBM·GRU 피크경보 앙상블

수치예측 모델은 02의 **LGBM 최종 잔차 보정 모델**로 확정했으므로 이 노트북에서는 다루지 않습니다.

피크경보 후보 두 개를 같은 7월 검증과 8~9월 개발 검증에서 결합합니다.

| 모델 | 입력과 학습 | 단독 성향 |
|---|---|---|
| LGBM 개선 Q1 | 개선 47개 입력, q=0.75, 월요일별 재학습 | 정밀도가 높고 오경보가 적음 |
| GRU 개선 모델 | 개선 47개 입력, q=0.80, 월요일별 재학습 | 재현율이 높고 미탐지가 적음 |

앙상블 예측값은 `(1-GRU 비중) × LGBM + GRU 비중 × GRU`로 계산하고, 모든 후보에 고정 기준 176을 적용합니다. GRU 비중은 0%부터 100%까지 5% 간격으로 비교합니다. 7~9월 통합 F1이 가장 높은 비율을 선택하고, 동률이면 두 기간 중 낮은 F1, 통합 정밀도, 통합 재현율, 낮은 GRU 비중 순서로 선택합니다.

8~9월도 모델 선택에 사용했으므로 결과는 **개발 검증**이며 별도의 독립 테스트 결과로 해석하지 않습니다.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd()
if not (ROOT / 'results/preprocessing_24h/config.json').exists():
    ROOT = ROOT.parent

PREP = ROOT / 'results/preprocessing_24h'
LGBM_FILES = {
    'validation_7월': ROOT / 'results/team_submission_runtime/outputs/validation_predictions_7월.csv',
    'development_8_9월': ROOT / 'results/team_submission_runtime/outputs/test_predictions_8_9월.csv',
}
GRU_FILES = {
    'validation_7월': ROOT / 'results/gru_24h_team_preprocessing/aligned_with_team_lgbm/aligned_predictions_validation_7월.csv',
    'development_8_9월': ROOT / 'results/gru_24h_team_preprocessing/aligned_with_team_lgbm/aligned_predictions_development_8_9월.csv',
}
GRU_SELECTION_FILE = (
    ROOT / 'results/gru_24h_team_preprocessing/aligned_with_team_lgbm/'
    'final_gru_selection_7_9월.json'
)
OUT = ROOT / 'results/ensemble_lgbm_gru_team_aligned'
OUT.mkdir(parents=True, exist_ok=True)

with (PREP / 'config.json').open(encoding='utf-8') as file:
    prep_config = json.load(file)
with GRU_SELECTION_FILE.open(encoding='utf-8') as file:
    gru_selection = json.load(file)

THRESHOLD = float(prep_config['peak_threshold'])
LGBM_COLUMN = 'q1_new'
GRU_COLUMN = 'gru_enhanced47_weekly_q080'
GRU_WEIGHTS = np.round(np.arange(0.0, 1.0001, 0.05), 2)
PERIODS = ['validation_7월', 'development_8_9월']
EXPECTED_ROWS = {'validation_7월': 60192, 'development_8_9월': 101472}

assert THRESHOLD == 176.0
assert gru_selection['selected_peak_gru'] == GRU_COLUMN
assert gru_selection['test_label'] == 'development_validation'
assert gru_selection['independent_test_remaining'] is False

print('LGBM 예측:', {k: str(v.resolve()) for k, v in LGBM_FILES.items()})
print('GRU 예측:', {k: str(v.resolve()) for k, v in GRU_FILES.items()})
print('GRU 최종 후보:', GRU_COLUMN)
print('피크 기준:', THRESHOLD)
print('GRU 비중 후보:', GRU_WEIGHTS.tolist())


## 1. 예측 불러오기와 일대일 정렬

각 기간에서 두 모델의 기준시각·예측대상시각·예측칸을 일대일로 맞춥니다. 실제 전력과 행 수가 다르면 실행을 중단합니다.


In [ ]:
keys = ['기준시각', '예측대상시각', '예측칸']
aligned_parts = []

for period in PERIODS:
    lgbm_raw = pd.read_csv(
        LGBM_FILES[period],
        usecols=keys + ['실제전력', LGBM_COLUMN],
        parse_dates=['기준시각', '예측대상시각'],
    )
    gru_raw = pd.read_csv(
        GRU_FILES[period],
        usecols=keys + ['실제전력', GRU_COLUMN],
        parse_dates=['기준시각', '예측대상시각'],
    )
    assert not lgbm_raw.duplicated(keys).any()
    assert not gru_raw.duplicated(keys).any()

    part = lgbm_raw.merge(
        gru_raw,
        on=keys,
        how='inner',
        suffixes=('_lgbm', '_gru'),
        validate='one_to_one',
    ).sort_values(keys, kind='stable').reset_index(drop=True)

    assert len(part) == len(lgbm_raw) == len(gru_raw) == EXPECTED_ROWS[period]
    np.testing.assert_allclose(
        part['실제전력_lgbm'].to_numpy(float),
        part['실제전력_gru'].to_numpy(float),
        rtol=0,
        atol=1e-6,
    )
    part.insert(0, 'period', period)
    aligned_parts.append(part)

aligned = pd.concat(aligned_parts, ignore_index=True)
aligned['actual'] = aligned['실제전력_lgbm'].astype(np.float64)
aligned['lgbm_prediction'] = aligned[LGBM_COLUMN].astype(np.float64)
aligned['gru_prediction'] = aligned[GRU_COLUMN].astype(np.float64)

assert len(aligned) == sum(EXPECTED_ROWS.values())
assert np.isfinite(aligned[['actual', 'lgbm_prediction', 'gru_prediction']]).all().all()

for period in PERIODS:
    part = aligned.loc[aligned['period'].eq(period)]
    print(
        period, ':', len(part), '칸 / 실제 피크',
        int(part['actual'].ge(THRESHOLD).sum()), '칸 / 기준시각',
        part['기준시각'].min(), '~', part['기준시각'].max(),
    )
print('통합:', len(aligned), '칸')


## 2. GRU 비중별 고정 176 평가

가중치 외에는 조건을 바꾸지 않습니다. 각 비율을 7월, 8~9월, 7~9월 통합으로 평가하고 통합 F1을 기준으로 하나의 비율을 선택합니다.


In [ ]:
def evaluate_peak(y_true, prediction, threshold=THRESHOLD):
    actual_peak = y_true >= threshold
    predicted_peak = prediction >= threshold
    tp = int((actual_peak & predicted_peak).sum())
    fp = int((~actual_peak & predicted_peak).sum())
    fn = int((actual_peak & ~predicted_peak).sum())
    tn = int((~actual_peak & ~predicted_peak).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    f2 = 5 * precision * recall / (4 * precision + recall) if precision + recall else 0.0
    error = prediction - y_true
    return {
        'observations': int(len(y_true)),
        'mae': float(np.abs(error).mean()),
        'rmse': float(np.sqrt(np.mean(error ** 2))),
        'bias': float(error.mean()),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f2': f2,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
    }

weight_rows = []
for gru_weight in GRU_WEIGHTS:
    for period in PERIODS + ['pooled_7_9월']:
        part = aligned if period == 'pooled_7_9월' else aligned.loc[aligned['period'].eq(period)]
        prediction = (
            (1.0 - gru_weight) * part['lgbm_prediction'].to_numpy()
            + gru_weight * part['gru_prediction'].to_numpy()
        )
        weight_rows.append({
            'period': period,
            'gru_weight': float(gru_weight),
            'lgbm_weight': float(1.0 - gru_weight),
            **evaluate_peak(part['actual'].to_numpy(), prediction),
        })
weight_results = pd.DataFrame(weight_rows)

period_min_f1 = (
    weight_results.loc[weight_results['period'].isin(PERIODS)]
    .groupby('gru_weight', as_index=False)['f1']
    .min()
    .rename(columns={'f1': 'minimum_period_f1'})
)
selection_table = (
    weight_results.loc[weight_results['period'].eq('pooled_7_9월')]
    .merge(period_min_f1, on='gru_weight', how='left', validate='one_to_one')
)
selected = selection_table.sort_values(
    ['f1', 'minimum_period_f1', 'precision', 'recall', 'gru_weight'],
    ascending=[False, False, False, False, True],
    kind='stable',
).iloc[0]
selected_gru_weight = float(selected['gru_weight'])
selected_lgbm_weight = float(selected['lgbm_weight'])
aligned['ensemble_prediction'] = (
    selected_lgbm_weight * aligned['lgbm_prediction']
    + selected_gru_weight * aligned['gru_prediction']
)

display(selection_table[[
    'lgbm_weight', 'gru_weight', 'precision', 'recall',
    'f1', 'minimum_period_f1', 'f2', 'fp', 'fn',
]].round(4))
print(
    f'선택 비율: LGBM {selected_lgbm_weight:.0%} + '
    f'GRU {selected_gru_weight:.0%}'
)


## 3. 단독 모델과 선택 앙상블 비교

LGBM 단독, GRU 단독, 선택된 앙상블을 각 기간과 통합 결과에서 비교합니다. 앙상블이 통합 F1 기준으로 단독 모델보다 좋은지도 확인합니다.


In [ ]:
model_columns = {
    'LGBM_q075': 'lgbm_prediction',
    'GRU_q080': 'gru_prediction',
    'ensemble': 'ensemble_prediction',
}
comparison_rows = []
for period in PERIODS + ['pooled_7_9월']:
    part = aligned if period == 'pooled_7_9월' else aligned.loc[aligned['period'].eq(period)]
    for model, column in model_columns.items():
        comparison_rows.append({
            'period': period,
            'model': model,
            'lgbm_weight': (
                1.0 if model == 'LGBM_q075'
                else 0.0 if model == 'GRU_q080'
                else selected_lgbm_weight
            ),
            'gru_weight': (
                0.0 if model == 'LGBM_q075'
                else 1.0 if model == 'GRU_q080'
                else selected_gru_weight
            ),
            **evaluate_peak(
                part['actual'].to_numpy(),
                part[column].to_numpy(),
            ),
        })
comparison = pd.DataFrame(comparison_rows)
pooled_comparison = comparison.loc[comparison['period'].eq('pooled_7_9월')]
single_best_f1 = pooled_comparison.loc[
    pooled_comparison['model'].ne('ensemble'), 'f1'
].max()
ensemble_f1 = float(
    pooled_comparison.loc[pooled_comparison['model'].eq('ensemble'), 'f1'].iloc[0]
)
ensemble_improves_f1 = bool(ensemble_f1 > single_best_f1 + 1e-12)

display(comparison[[
    'period', 'model', 'lgbm_weight', 'gru_weight',
    'precision', 'recall', 'f1', 'f2', 'fp', 'fn',
]].round(4))
print('통합 단독 최고 F1:', round(float(single_best_f1), 4))
print('통합 앙상블 F1:', round(ensemble_f1, 4))
print('앙상블이 단독 최고 F1을 초과:', ensemble_improves_f1)


## 4. 결과 저장과 시각화

기간별 가중치 결과, 모델 비교표, 통합 예측값과 선택 근거를 저장합니다.


In [ ]:
weight_results.to_csv(
    OUT / 'weight_comparison_all_periods.csv',
    index=False,
    encoding='utf-8-sig',
)
comparison.to_csv(
    OUT / 'model_comparison_all_periods.csv',
    index=False,
    encoding='utf-8-sig',
)

prediction_output = aligned[['period'] + keys].copy()
prediction_output['actual'] = aligned['actual']
prediction_output['lgbm_q075'] = aligned['lgbm_prediction']
prediction_output['gru_q080'] = aligned['gru_prediction']
prediction_output['ensemble'] = aligned['ensemble_prediction']
prediction_output['threshold'] = THRESHOLD
prediction_output['actual_peak'] = aligned['actual'].ge(THRESHOLD).astype(np.int8)
prediction_output['lgbm_alarm'] = aligned['lgbm_prediction'].ge(THRESHOLD).astype(np.int8)
prediction_output['gru_alarm'] = aligned['gru_prediction'].ge(THRESHOLD).astype(np.int8)
prediction_output['ensemble_alarm'] = aligned['ensemble_prediction'].ge(THRESHOLD).astype(np.int8)
prediction_output.to_csv(
    OUT / 'predictions_all_7_9월.csv',
    index=False,
    encoding='utf-8-sig',
)

selection = {
    'evaluation_period': '2021-07 validation + 2021-08~09 development validation',
    'evaluation_cells': int(len(aligned)),
    'actual_peak_threshold': THRESHOLD,
    'selection_metric': 'pooled F1; tie: minimum period F1, precision, recall, lower GRU weight',
    'weight_candidates': GRU_WEIGHTS.tolist(),
    'lgbm_model': 'improved_Q1_q075',
    'lgbm_prediction_column': LGBM_COLUMN,
    'lgbm_sources': {key: str(value) for key, value in LGBM_FILES.items()},
    'gru_model': 'enhanced_47_features_q080',
    'gru_prediction_column': GRU_COLUMN,
    'gru_sources': {key: str(value) for key, value in GRU_FILES.items()},
    'selected_lgbm_weight': selected_lgbm_weight,
    'selected_gru_weight': selected_gru_weight,
    'selected_metrics_pooled': {
        key: (
            int(selected[key])
            if key in ['observations', 'tp', 'fp', 'fn', 'tn']
            else float(selected[key])
        )
        for key in [
            'observations', 'mae', 'rmse', 'bias',
            'precision', 'recall', 'f1', 'f2',
            'tp', 'fp', 'fn', 'tn',
        ]
    },
    'minimum_period_f1': float(selected['minimum_period_f1']),
    'single_best_f1_pooled': float(single_best_f1),
    'ensemble_improves_single_best_f1': ensemble_improves_f1,
    'test_used_for_selection': True,
    'test_label': 'development_validation',
    'independent_test_remaining': False,
}
(OUT / 'selection_all_7_9월.json').write_text(
    json.dumps(selection, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

pooled_weights = weight_results.loc[weight_results['period'].eq('pooled_7_9월')]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for metric, label in [('f1', 'F1'), ('precision', '정밀도'), ('recall', '재현율')]:
    axes[0].plot(pooled_weights['gru_weight'], pooled_weights[metric], marker='o', label=label)
axes[0].axvline(selected_gru_weight, color='black', linestyle='--', label='선택 비율')
axes[0].set(xlabel='GRU 비중', ylabel='점수', title='7~9월 통합 피크경보 성능')
axes[0].legend()

axes[1].plot(pooled_weights['gru_weight'], pooled_weights['fp'], marker='o', label='오경보')
axes[1].plot(pooled_weights['gru_weight'], pooled_weights['fn'], marker='o', label='미탐지')
axes[1].axvline(selected_gru_weight, color='black', linestyle='--', label='선택 비율')
axes[1].set(xlabel='GRU 비중', ylabel='건수', title='7~9월 통합 오경보와 미탐지')
axes[1].legend()

fig.tight_layout()
fig.savefig(OUT / 'ensemble_weight_comparison_all_7_9월.png', dpi=160, bbox_inches='tight')
plt.show()

print('저장 완료:', OUT.resolve())
